# Monthyl Core HR metrics

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact

### Globals

In [2]:
current_month = pd.to_datetime('2024-11-01')
current_month_formatted = current_month.strftime("%b '%y")
previous_month = current_month - pd.DateOffset(months=1)
previous_month_formatted = previous_month.strftime("%b '%y")


In [3]:
current_month.strftime('%Y-%m')

'2024-11'

## Data preparation

In [4]:
# declaring fields in use
fields = [
    'ds_start', 'ds', 'is_last_day_of_year', 'employee_id', 'hire_date', 'prehire_status', 'is_active',
    'termination_date', 'termination_reason', 'is_termination_voluntary', 'is_terminated',
    'org_l00', 'org_l01', 'org_l02', 'org_l03', 
    'gender', 'gender_remapped', 'ethnicity', 'ethnicity_remapped', 
    'is_manager_track', 'job_track', 'job_level_idx',
    'job_level_category', 'job_level_category_ordered_w_indicators'
]

### Monthly employee data

In [5]:
df_employees_plus = pd.read_csv(
    filepath_or_buffer='../transforms/exclude/employee_data_plus.csv',
    dtype='str'
)
# type casting and renaming fields
df_employees_plus['ds'] = pd.to_datetime(df_employees_plus['full_date']).dt.normalize()
df_employees_plus['ds_start'] = df_employees_plus.ds.dt.to_period('M').dt.start_time
df_employees_plus['is_last_day_of_year'] = df_employees_plus.is_last_day_of_year.astype('bool')
df_employees_plus['hire_date'] = pd.to_datetime(df_employees_plus['hire_date']).dt.normalize()
df_employees_plus['termination_date'] = pd.to_datetime(df_employees_plus['termination_date_coalesced']).dt.normalize()
df_employees_plus['is_termination_voluntary'] = df_employees_plus.is_termination_voluntary.astype('bool')
df_employees_plus['is_manager_track'] = df_employees_plus.is_manager_track.astype('bool')

# deriving is_active
condition_active = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds <= df_employees_plus.termination_date)
df_employees_plus['is_active'] = condition_active

# deriving is_terminated
condition_terminated_in_current_month = (df_employees_plus.ds >= df_employees_plus.hire_date) &\
    (df_employees_plus.ds_start <= df_employees_plus.termination_date) &\
    (df_employees_plus.ds >= df_employees_plus.termination_date)
df_employees_plus['is_terminated'] = condition_terminated_in_current_month

# reorganizing fields
df_employees_plus = df_employees_plus[fields]

df_employees_plus.dtypes

ds_start                                   datetime64[ns]
ds                                         datetime64[ns]
is_last_day_of_year                                  bool
employee_id                                        object
hire_date                                  datetime64[ns]
prehire_status                                     object
is_active                                            bool
termination_date                           datetime64[ns]
termination_reason                                 object
is_termination_voluntary                             bool
is_terminated                                        bool
org_l00                                            object
org_l01                                            object
org_l02                                            object
org_l03                                            object
gender                                             object
gender_remapped                                    object
ethnicity     

In [6]:
df_employees_plus.ds.dt.to_period('Y').dt.end_time.dt.date

0         2021-12-31
1         2021-12-31
2         2021-12-31
3         2021-12-31
4         2021-12-31
             ...    
412195    2026-12-31
412196    2026-12-31
412197    2026-12-31
412198    2026-12-31
412199    2026-12-31
Name: ds, Length: 412200, dtype: object

In [7]:
condition_new_hire = (df_employees_plus.hire_date >= df_employees_plus.ds_start) &\
    (df_employees_plus.hire_date <= df_employees_plus.ds)
df_employees_plus['is_new_hire'] = condition_new_hire

In [8]:
df_employees_plus.head()

,ds_start,ds,is_last_day_of_year,employee_id,hire_date,prehire_status,is_active,termination_date,termination_reason,is_termination_voluntary,...,gender,gender_remapped,ethnicity,ethnicity_remapped,is_manager_track,job_track,job_level_idx,job_level_category,job_level_category_ordered_w_indicators,is_new_hire
0,2021-01-01,2021-01-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
1,2021-02-01,2021-02-28,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
2,2021-03-01,2021-03-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
3,2021-04-01,2021-04-30,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False
4,2021-05-01,2021-05-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11),False


### Annual employee data

In [9]:
# extract and derive initial fields ------------------------------------------------
df_employees_plus_annual = pd.read_csv(
    filepath_or_buffer='../transforms/exclude/employee_data_plus_annual.csv',
    dtype='str'
)
# type casting and renaming fields
df_employees_plus_annual['ds'] = pd.to_datetime(df_employees_plus_annual['full_date']).dt.normalize()
df_employees_plus_annual['ds_start'] = df_employees_plus_annual.ds.dt.to_period('Y').dt.start_time
df_employees_plus_annual['is_last_day_of_year'] = df_employees_plus_annual.is_last_day_of_year.astype('bool')
df_employees_plus_annual['hire_date'] = pd.to_datetime(df_employees_plus_annual['hire_date']).dt.normalize()
df_employees_plus_annual['termination_date'] = pd.to_datetime(df_employees_plus_annual['termination_date_coalesced']).dt.normalize()
df_employees_plus_annual['is_termination_voluntary'] = df_employees_plus_annual.is_termination_voluntary.astype('bool')
df_employees_plus_annual['is_manager_track'] = df_employees_plus_annual.is_manager_track.astype('bool')

# deriving is_active
condition_active = (df_employees_plus_annual.ds >= df_employees_plus_annual.hire_date) &\
    (df_employees_plus_annual.ds <= df_employees_plus_annual.termination_date)
df_employees_plus_annual['is_active'] = condition_active

# deriving is_terminated
condition_terminated_in_current_month = (df_employees_plus_annual.ds >= df_employees_plus_annual.hire_date) &\
    (df_employees_plus_annual.ds_start <= df_employees_plus_annual.termination_date) &\
    (df_employees_plus_annual.ds >= df_employees_plus_annual.termination_date)
df_employees_plus_annual['is_terminated'] = condition_terminated_in_current_month

# reorganizing fields
df_employees_plus_annual = df_employees_plus_annual[fields]

df_employees_plus_annual.dtypes

# is_new_hire ----------------------------------------------------------------------
condition_new_hire = (df_employees_plus.hire_date >= df_employees_plus.ds_start) &\
    (df_employees_plus.hire_date <= df_employees_plus.ds)
df_employees_plus['is_new_hire'] = condition_new_hire


In [10]:
df_employees_plus_annual

,ds_start,ds,is_last_day_of_year,employee_id,hire_date,prehire_status,is_active,termination_date,termination_reason,is_termination_voluntary,...,org_l03,gender,gender_remapped,ethnicity,ethnicity_remapped,is_manager_track,job_track,job_level_idx,job_level_category,job_level_category_ordered_w_indicators
0,2021-01-01,2021-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
1,2022-01-01,2022-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
2,2023-01-01,2023-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
3,2024-01-01,2024-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
4,2025-01-01,2025-12-31,True,e000001,2014-01-02,Not prehire,True,2260-01-01,NaN,True,...,NaN,Male,Male,White,00--White,True,M,11,SVP,10--SVP (M11)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34345,2022-01-01,2022-12-31,True,e005725,2024-11-25,Prehire,False,2260-01-01,NaN,True,...,Consumer Direct,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3)
34346,2023-01-01,2023-12-31,True,e005725,2024-11-25,Prehire,False,2260-01-01,NaN,True,...,Consumer Direct,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3)
34347,2024-01-01,2024-12-31,True,e005725,2024-11-25,Not prehire,True,2260-01-01,NaN,True,...,Consumer Direct,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3)
34348,2025-01-01,2025-12-31,True,e005725,2024-11-25,Not prehire,True,2260-01-01,NaN,True,...,Consumer Direct,Male,Male,Hispanic,02--Minority,True,IC,3,Associate,02--Associate (IC1-3)


In [11]:
df_employees_plus_annual[(df_employees_plus_annual.employee_id == 'e000029')
                        #  & (df_employees_plus_annual.ds == '2024-12-31')
                         & (True)] \
    [['ds_start', 'ds', 'employee_id', 'hire_date', 'termination_date']]

,ds_start,ds,employee_id,hire_date,termination_date
168,2021-01-01,2021-12-31,e000029,2014-06-23,2024-03-06
169,2022-01-01,2022-12-31,e000029,2014-06-23,2024-03-06
170,2023-01-01,2023-12-31,e000029,2014-06-23,2024-03-06
171,2024-01-01,2024-12-31,e000029,2014-06-23,2024-03-06
172,2025-01-01,2025-12-31,e000029,2014-06-23,2024-03-06
173,2026-01-01,2026-12-31,e000029,2014-06-23,2024-03-06


## Monthly results

### Creating monthly dataframes

#### Monthly (overall)

In [12]:
df_monthly = pd.DataFrame({'ds': df_employees_plus.ds.unique(), 'ds_start': df_employees_plus.ds_start.unique()})


##### Deriving `n_active_employees`

In [13]:
# df_monthly['n_active_employees_by_status']
active_employees_by_status = df_employees_plus[df_employees_plus['is_active']] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_active_employees')
df_monthly = df_monthly.merge(right=active_employees_by_status, how='left', on='ds')

##### Deriving `n_monthly_terminated_employees`

In [14]:
terminated_by_dates_monthly_by_status = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_monthly_terminated_employees')

df_monthly = df_monthly.merge(right=terminated_by_dates_monthly_by_status, how='left', on='ds')
df_monthly['n_monthly_terminated_employees'] = pd.to_numeric(df_monthly.n_monthly_terminated_employees, errors='coerce').fillna(0).astype('int')


##### Deriving fields for `attrition_rate`

- `avg_active_employees` - active employee count 2 month rolling (between current month and previous month)
- `attrition_rate = terminated_employees / avg_active_employees`

In [15]:
df_monthly['avg_active_employees'] = df_monthly['n_active_employees'].rolling(window=2).mean()
df_monthly['avg_active_employees'] = np.where(
    df_monthly['avg_active_employees'].isnull(),
    df_monthly['n_active_employees'],
    df_monthly['avg_active_employees']
)
df_monthly['attrition_rate'] = df_monthly['n_monthly_terminated_employees'] / df_monthly['avg_active_employees']

##### Deriving `n_new_hire_employees`

In [16]:
new_hire_employees_by_status = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')
df_monthly = df_monthly.merge(right=new_hire_employees_by_status, how='left', on='ds', )
df_monthly['n_new_hire_employees'] = df_monthly['n_new_hire_employees'].fillna(0).astype('int')

#### Monthly results by organization

##### Monthly by org_l01

Deriving counts for active, terminated, and new hire employees

In [17]:
temp_indices = ['ds', 'ds_start', 'org_l00', 'org_l01']

# deriving active employee count
df_monthly_by_org1 = df_employees_plus[df_employees_plus['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# adding placeholder field and reordering fields
df_monthly_by_org1['org_l02'] = df_monthly_by_org1['org_l01'] + ' Total'
df_monthly_by_org1 = df_monthly_by_org1[temp_indices + ['org_l02'] + ['n_active_employees']]

# deriving new hire employee count
new_hires_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_monthly_terminated_employees') \

# merging and typecasting
df_monthly_by_org1 = pd.merge(left=df_monthly_by_org1, right=new_hires_by_dates_monthly_by_status_and_org, how='left', on=temp_indices)
df_monthly_by_org1['n_new_hire_employees'] = pd.to_numeric(df_monthly_by_org1.n_new_hire_employees, errors='coerce')
df_monthly_by_org1['n_new_hire_employees'] = df_monthly_by_org1.n_new_hire_employees.fillna(0).astype('int')

df_monthly_by_org1 = pd.merge(left=df_monthly_by_org1, right=terminated_by_dates_monthly_by_status_and_org, how='left', on=temp_indices)
df_monthly_by_org1['n_monthly_terminated_employees'] = pd.to_numeric(df_monthly_by_org1['n_monthly_terminated_employees'], errors='coerce')
df_monthly_by_org1['n_monthly_terminated_employees'] = df_monthly_by_org1['n_monthly_terminated_employees'].fillna(0).astype('int')


Deriving two-month average of active employees

In [18]:
df_temp_reindexed = df_monthly_by_org1 \
    .set_index(temp_indices).sort_index(level='ds') \
    .copy()
df_temp_avg_active_employees = df_temp_reindexed.groupby(level=temp_indices[2:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0,1], drop=True) \
    .reset_index(name='avg_active_employees')

df_monthly_by_org1 = pd.merge(left=df_monthly_by_org1, right=df_temp_avg_active_employees, how='left', on=temp_indices)
df_monthly_by_org1['avg_active_employees'] = np.where(
    df_monthly_by_org1['avg_active_employees'].isnull(),
    df_monthly_by_org1['n_active_employees'],
    df_monthly_by_org1['avg_active_employees']
)

Deriving `attrition_rate`

In [19]:
df_monthly_by_org1['attrition_rate'] = df_monthly_by_org1['n_monthly_terminated_employees'] / df_monthly_by_org1['avg_active_employees']

##### Monthly by org_l02

Deriving counts for active, terminated, and new hire employees

In [20]:
temp_indices = ['ds', 'ds_start', 'org_l00', 'org_l01', 'org_l02']

# deriving active employee count
df_monthly_by_org2 = df_employees_plus[df_employees_plus['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# deriving new hire employee count
new_hires_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_org = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_monthly_terminated_employees') \

df_monthly_by_org2 = pd.merge(left=df_monthly_by_org2, right=new_hires_by_dates_monthly_by_status_and_org, how='left', on=temp_indices)
df_monthly_by_org2['n_new_hire_employees'] = pd.to_numeric(df_monthly_by_org2.n_new_hire_employees, errors='coerce')
df_monthly_by_org2['n_new_hire_employees'] = df_monthly_by_org2.n_new_hire_employees.fillna(0).astype('int')

df_monthly_by_org2 = pd.merge(left=df_monthly_by_org2, right=terminated_by_dates_monthly_by_status_and_org, how='left', on=temp_indices)
df_monthly_by_org2['n_monthly_terminated_employees'] = pd.to_numeric(df_monthly_by_org2['n_monthly_terminated_employees'], errors='coerce')
df_monthly_by_org2['n_monthly_terminated_employees'] = df_monthly_by_org2['n_monthly_terminated_employees'].fillna(0).astype('int')


Deriving two-month average of active employees

In [21]:
df_temp_reindexed = df_monthly_by_org2 \
    .set_index(temp_indices).sort_index(level='ds') \
    .copy()
df_temp_avg_active_employees = df_temp_reindexed.groupby(level=temp_indices[2:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0,1,2], drop=True) \
    .reset_index(name='avg_active_employees')

df_monthly_by_org2 = pd.merge(left=df_monthly_by_org2, right=df_temp_avg_active_employees, how='left', on=temp_indices)
df_monthly_by_org2['avg_active_employees'] = np.where(
    df_monthly_by_org2['avg_active_employees'].isnull(),
    df_monthly_by_org2['n_active_employees'],
    df_monthly_by_org2['avg_active_employees']
)

Deriving `attrition_rate`

In [22]:
df_monthly_by_org2['attrition_rate'] = df_monthly_by_org2['n_monthly_terminated_employees'] / df_monthly_by_org2['avg_active_employees']

#### Monthly results by level

Deriving the following by level
- `n_active_employees`, `n_new_hire_employees`, `n_monthly_terminated_employees`, `avg_active_employees`, `job_level_index`, `job_level_label`

In [23]:
temp_indices = ['ds', 'ds_start', 'job_level_category_ordered_w_indicators']

# deriving active employee count
df_monthly_by_level = df_employees_plus[df_employees_plus['is_active']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_active_employees') \

# deriving new hire employee count
new_hires_by_dates_monthly_by_status_and_level = df_employees_plus[df_employees_plus['is_new_hire']] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_new_hire_employees')

# deriving terminated employee count
terminated_by_dates_monthly_by_status_and_level = df_employees_plus[df_employees_plus.is_terminated] \
    .groupby(temp_indices)['employee_id'].nunique() \
    .reset_index(name='n_monthly_terminated_employees')

df_monthly_by_level = pd.merge(left=df_monthly_by_level, right=new_hires_by_dates_monthly_by_status_and_level, how='left', on=temp_indices)
df_monthly_by_level['n_new_hire_employees'] = pd.to_numeric(df_monthly_by_level.n_new_hire_employees, errors='coerce')
df_monthly_by_level['n_new_hire_employees'] = df_monthly_by_level.n_new_hire_employees.fillna(0).astype('int')

df_monthly_by_level = pd.merge(left=df_monthly_by_level, right=terminated_by_dates_monthly_by_status_and_level, how='left', on=temp_indices)
df_monthly_by_level['n_monthly_terminated_employees'] = pd.to_numeric(df_monthly_by_level['n_monthly_terminated_employees'], errors='coerce')
df_monthly_by_level['n_monthly_terminated_employees'] = df_monthly_by_level['n_monthly_terminated_employees'].fillna(0).astype('int')

df_temp_reindexed = df_monthly_by_level \
    .set_index(temp_indices).sort_index(level='ds') \
    .copy()
df_temp_avg_active_employees = df_temp_reindexed.groupby(level=temp_indices[2:]) \
    ['n_active_employees'] \
    .rolling(window=2).mean() \
    .reset_index(level=[0], drop=True) \
    .reset_index(name='avg_active_employees')

df_monthly_by_level = pd.merge(left=df_monthly_by_level, right=df_temp_avg_active_employees, how='left', on=temp_indices)
df_monthly_by_level['avg_active_employees'] = np.where(
    df_monthly_by_level['avg_active_employees'].isnull(),
    df_monthly_by_level['n_active_employees'],
    df_monthly_by_level['avg_active_employees']
)

df_monthly_by_level[['job_level_index', 'job_level_label']] = df_monthly_by_level.job_level_category_ordered_w_indicators.str.split('--', n=1, expand=True)


## Annual results

### Annual (overall)

In [24]:
df_annual = pd.DataFrame({
    'ds': df_employees_plus_annual[df_employees_plus_annual.is_last_day_of_year].ds.unique(), 
}).sort_values('ds')

df_annual['ds_start'] = df_annual.ds.dt.to_period('Y').dt.start_time


Deriving `n_active_employees`

In [25]:
active_employees_by_status = df_employees_plus_annual[df_employees_plus_annual['is_active']] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_active_employees')
df_annual = df_annual.merge(right=active_employees_by_status, how='left', on='ds')


##### Deriving `n_monthly_terminated_employees`

In [26]:
terminated_by_dates_annual_by_status = df_employees_plus_annual[df_employees_plus_annual.is_terminated] \
    .groupby('ds')['employee_id'].nunique() \
    .reset_index(name='n_annual_terminated_employees')

df_annual = df_annual.merge(right=terminated_by_dates_annual_by_status, how='left', on='ds')
df_annual['n_annual_terminated_employees'] = pd.to_numeric(df_annual.n_annual_terminated_employees, errors='coerce').fillna(0).astype('int')


In [27]:
df_annual

,ds,ds_start,n_active_employees,n_annual_terminated_employees
0,2021-12-31,2021-01-01,1457,347
1,2022-12-31,2022-01-01,1824,633
2,2023-12-31,2023-01-01,1943,838
3,2024-12-31,2024-01-01,2014,687
4,2025-12-31,2025-01-01,2014,0
5,2026-12-31,2026-01-01,2014,0


## Attrition rate significance analysis

<summary>Monthly attrition significance POC (proof of concept)</summary>
<details>
    <code>
        df_2021_01_31 = df_monthly_by_org2[
            (df_monthly_by_org2.ds == '2021-01-31')
        ].sort_values('org_l02') \
            [['ds', 'org_l01', 'org_l02', 'n_active_employees', 'n_monthly_terminated_employees', 'attrition_rate']] \
            .reset_index(drop=True)
        df_2021_01_31['n_not_terminated_employees'] = df_2021_01_31['n_active_employees'] - df_2021_01_31['n_monthly_terminated_employees']

        total_terminated = df_2021_01_31['n_monthly_terminated_employees'].sum()
        total_not_terminated = df_2021_01_31['n_not_terminated_employees'].sum()

        results = []

        for index, row in df_2021_01_31.iterrows():
            # for org under review
            org_name = row['org_l02']
            terminated_org = row['n_monthly_terminated_employees']
            not_terminated_org = row['n_not_terminated_employees']
            
            # for other orgs
            terminated_rest = total_terminated - terminated_org
            not_terminated_rest = total_not_terminated - not_terminated_org
            
            # 2x2 contingency table
            # (org vs others) x (terminated vs not terminated)
            table = [[terminated_org, not_terminated_org],
                    [terminated_rest, not_terminated_rest]]
            
            odds_ratio, p_value = fisher_exact(table)
            
            results.append({
                'org_l02': org_name,
                'contingency_table': str(table),
                'odds_ratio': odds_ratio,
                'p_value': p_value
            })
            
        results_df = pd.DataFrame(results)
        results_df.sort_values(by='p_value')
    </code>
</details>

In [28]:
df_monthly_by_org_l02 = df_monthly_by_org2.copy(True)

df_monthly_by_org_l02['n_not_terminated_employees'] = df_monthly_by_org_l02['n_active_employees'] - df_monthly_by_org_l02['n_monthly_terminated_employees']

df_monthly_by_org_l02['contingency_table'] = ''
df_monthly_by_org_l02['odds_ratio'] = np.NaN
df_monthly_by_org_l02['p_value'] = np.NaN

for ds in df_monthly_by_org_l02.ds.unique():
    df_current = df_monthly_by_org_l02[df_monthly_by_org_l02.ds == ds].copy(True)
    total_terminated = df_current['n_monthly_terminated_employees'].sum()
    total_not_terminated = df_current['n_not_terminated_employees'].sum()

    for index, row in df_current.iterrows():
        # for org under review
        ds = row['ds']
        org_name = row['org_l02']
        terminated_org = row['n_monthly_terminated_employees']
        not_terminated_org = row['n_not_terminated_employees']
        
        # for other orgs
        terminated_rest = total_terminated - terminated_org
        not_terminated_rest = total_not_terminated - not_terminated_org
        
        # 2x2 contingency table
        # (org vs others) x (terminated vs not terminated)
        table = [[terminated_org, not_terminated_org],
                [terminated_rest, not_terminated_rest]]
        
        odds_ratio, p_value = fisher_exact(table)
        
        contingency_table = str(table)
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'contingency_table'
        ] = str(table)
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'odds_ratio'
        ] = odds_ratio
        
        df_monthly_by_org_l02.loc[
            (df_monthly_by_org_l02.ds == ds)
            & (df_monthly_by_org_l02.org_l02 == org_name), 
            'p_value'
        ] = p_value
df_monthly_by_org_l02['is_significant'] = df_monthly_by_org_l02['p_value'] <= 0.05

In [29]:
ds = '2022-02-28'
df_monthly_by_org_l02 \
    [df_monthly_by_org_l02.ds == ds] \
    [['ds', 'org_l00', 'org_l01', 'org_l02', 'attrition_rate', 'p_value', 'is_significant']] \
    .sort_values(['org_l00', 'org_l01', 'org_l02']) \
    .reset_index(drop=True) \
    .style.format({
        'ds': lambda t: t.strftime('%Y-%m-%d') if pd.notnull(t) else '',
        'attrition_rate': '{:.1%}',
        'p_value': '{:.2f}'
    })

,ds,org_l00,org_l01,org_l02,attrition_rate,p_value,is_significant
0,2022-02-28,Company Inc.,Administrative,Communications,8.9%,0.23,False
1,2022-02-28,Company Inc.,Administrative,Finance,4.6%,0.80,False
2,2022-02-28,Company Inc.,Administrative,IT Services,12.3%,0.03,True
3,2022-02-28,Company Inc.,Administrative,Legal,8.2%,0.59,False
4,2022-02-28,Company Inc.,Administrative,People,2.6%,0.23,False
5,2022-02-28,Company Inc.,Administrative,Risk Management,5.3%,1.00,False
6,2022-02-28,Company Inc.,Production,Hardware,2.1%,0.08,False
7,2022-02-28,Company Inc.,Production,Quality Control,8.0%,0.49,False
8,2022-02-28,Company Inc.,Production,Research,2.5%,0.23,False
9,2022-02-28,Company Inc.,Production,Service Delivery,6.1%,1.00,False


## Report build

### Headcount over time

In [30]:
report_n_months = 24

mask_ds = df_monthly_by_org2.ds.isin(df_monthly_by_org2[df_monthly_by_org2.ds_start <= current_month] \
    .ds.unique()[-report_n_months:])

df_monthly_by_org2[mask_ds][['ds', 'n_active_employees', 'n_new_hire_employees', 'n_monthly_terminated_employees']] \
    .groupby(by=['ds']).sum() \
    .reset_index(drop=False) \
    .style.format({
        'ds': lambda t: t.strftime('%Y-%m-%d') if pd.notnull(t) else '',
        'n_active_employees': '{:,}'
    })


,ds,n_active_employees,n_new_hire_employees,n_monthly_terminated_employees
0,2022-12-31,"1,823",105,73
1,2023-01-31,"1,882",85,29
2,2023-02-28,"1,920",119,84
3,2023-03-31,"1,822",8,103
4,2023-04-30,"1,725",9,103
5,2023-05-31,"1,681",63,108
6,2023-06-30,"1,662",83,106
7,2023-07-31,"1,685",95,72
8,2023-08-31,"1,773",93,0
9,2023-09-30,"1,849",138,62


### Current month headcount by org

In [31]:
# defining orgs that show org_l02 breakdown and order of iteration
org_l01 = {'Sales': True, 'Production': True, 'Administrative': False}

# filtering data to current month, with corresponding org indices of interest
current_month_headcount_by_org = df_monthly_by_org2[df_monthly_by_org2.ds_start == current_month][['org_l01', 'org_l02', 'n_active_employees']].groupby(by=['org_l01', 'org_l02']).sum().sort_values(by=['org_l01', 'org_l02']).reset_index()

current_month_headcount_by_org['percent_of_total'] = current_month_headcount_by_org['n_active_employees'] / current_month_headcount_by_org['n_active_employees'].sum()

# creating blank dataframe
display_headcount_by_org = pd.DataFrame([{'org_l01': '', 'org_l02': '', 'n_active_employees': 0, 'percent_of_total': 0}])

# iterate through orgs determined in the dictionary above, appending subtotals where necessary
for org in org_l01:
    if org_l01[org]:
        display_headcount_by_org = pd.concat([display_headcount_by_org, current_month_headcount_by_org[current_month_headcount_by_org.org_l01 == org]], ignore_index=True)
    org_n_active_employees = current_month_headcount_by_org[current_month_headcount_by_org.org_l01 == org].n_active_employees.sum()
    total_active_employees = df_employees_plus[(df_employees_plus.ds_start == current_month) & (df_employees_plus.is_active)].employee_id.nunique()
    percent_of_total = org_n_active_employees / total_active_employees
    subtotal_row = {'org_l01': f'{org} Total', 'org_l02': f'{org} Total', 'n_active_employees': org_n_active_employees, 'percent_of_total': org_n_active_employees / total_active_employees}
    display_headcount_by_org = pd.concat([display_headcount_by_org, pd.DataFrame([subtotal_row])], ignore_index=True)

total_row = {'org_l01': 'Company Total', 'org_l02': 'Company Total', 'n_active_employees': total_active_employees, 'percent_of_total': 1}

# dropping record of blank dataframe
display_headcount_by_org.drop(index=display_headcount_by_org[(display_headcount_by_org.org_l01 == '') & (display_headcount_by_org.org_l02 == '')].index, axis=0, inplace=True)

# adding total record
display_headcount_by_org = pd.concat([display_headcount_by_org, pd.DataFrame([total_row])], ignore_index=True)

def format_and_combine(row):
    n_active_formatted = f"{row['n_active_employees']:,}"
    if row['org_l02'] == 'Company Total':
        return f"{n_active_formatted}"
    perc_formatted = f"{row['percent_of_total']:.1%}"
    
    return f"{n_active_formatted} ({perc_formatted})"

# applying formatting to a single column
display_headcount_by_org['headcount_and_percent_of_total'] = display_headcount_by_org.apply(format_and_combine, axis=1)

# drop and rename columns
display_headcount_by_org.rename(columns={'org_l02': 'org'}, inplace=True)
display_headcount_by_org.drop(columns=['org_l01', 'n_active_employees', 'percent_of_total'], inplace=True)

display_headcount_by_org


,org,headcount_and_percent_of_total
0,Business,155 (7.7%)
1,Consumer,154 (7.7%)
2,Retail,156 (7.7%)
3,Sales Operations,177 (8.8%)
4,Sales Training,158 (7.8%)
5,Sales Total,800 (39.7%)
6,Hardware,133 (6.6%)
7,Quality Control,107 (5.3%)
8,Research,109 (5.4%)
9,Service Delivery,122 (6.1%)


### Headcount year-over-year

In [32]:
df_headcount_monthly_yoy = df_monthly.copy(True)
df_headcount_monthly_yoy['yyyy'] = df_headcount_monthly_yoy['ds'].dt.year
df_headcount_monthly_yoy['mm'] = df_headcount_monthly_yoy['ds'].dt.month
df_headcount_monthly_yoy['mmm'] = df_headcount_monthly_yoy['ds'].dt.strftime('%b')

mask_ds = (df_headcount_monthly_yoy.ds_start <= current_month) \
    & (df_headcount_monthly_yoy.ds_start >= '2022-01-01')

df_pivoted = df_headcount_monthly_yoy[mask_ds] \
    [['yyyy', 'mm', 'mmm', 'ds', 'ds_start', 'n_active_employees', 'avg_active_employees', 'attrition_rate']] \
    .pivot_table(index='yyyy', columns='mm', values='n_active_employees')

# create a mapping from month number to abbreviation then rename the columns
month_map = {i: pd.Timestamp(f'2023-{i:02d}-01').strftime('%b') for i in df_pivoted.columns}
df_pivoted = df_pivoted.rename(columns=month_map)

# create formatting for each column then display
# not necessary when displaying into plot
def num_or_blank(x):
    return '' if pd.isnull(x) else f'{int(x):,}'

formatters = {
    col: num_or_blank if str(col) else None
    for col in df_pivoted.columns
}

df_pivoted.style.format(formatters)


mm,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
yyyy,,,,,,,,,,,,
2022,"1,411","1,517","1,581","1,599","1,604","1,602","1,559","1,564","1,748","1,716","1,796","1,824"
2023,"1,883","1,921","1,823","1,726","1,682","1,663","1,686","1,774","1,850","1,891","1,966","1,943"
2024,"1,994","1,907","1,878","1,802","1,838","1,825","1,870","1,930","2,048","2,034","2,014",


### Current month headcount by level

In [33]:
temp_indices = ['job_level_category_ordered_w_indicators', 'job_level_index', 'job_level_label']
# filtering data to current month, with corresponding org indices of interest
current_month_headcount_by_level = df_monthly_by_level[df_monthly_by_level.ds_start == current_month] \
    [temp_indices + ['n_active_employees']] \
    .groupby(by=temp_indices).sum() \
    .sort_values(by=temp_indices).reset_index()

current_month_headcount_by_level['percent_of_total'] = current_month_headcount_by_level['n_active_employees'] / current_month_headcount_by_level['n_active_employees'].sum()

total_active_employees = df_employees_plus[(df_employees_plus.ds_start == current_month) & (df_employees_plus.is_active)].employee_id.nunique()
total_row = {
    'job_level_category_ordered_w_indicators': 'Company Total', 
    'job_level_index': 'Company Total', 
    'job_level_label': 'Company Total', 
    'n_active_employees': total_active_employees, 
    'percent_of_total': 1
}

current_month_headcount_by_level = pd.concat([current_month_headcount_by_level, pd.DataFrame([total_row])], ignore_index=True)

def format_and_combine(row):
    n_active_formatted = f"{row['n_active_employees']:,}"
    if row['job_level_label'] == 'Company Total':
        return f"{n_active_formatted}"
    perc_formatted = f"{row['percent_of_total']:.1%}"
    
    return f"{n_active_formatted} ({perc_formatted})"

# applying formatting to a single column
current_month_headcount_by_level['headcount_and_percent_of_total'] = current_month_headcount_by_level.apply(format_and_combine, axis=1)

current_month_headcount_by_level.rename(columns={
    'job_level_label': 'Level',
    'headcount_and_percent_of_total': 'Headcount'
})[['Level', 'Headcount']]


,Level,Headcount
0,Support (S1-3),527 (26.2%)
1,Associate (IC1-3),578 (28.7%)
2,Senior (IC4-5),285 (14.2%)
3,Staff (IC 6-7),267 (13.3%)
4,Principal (IC 8-10),50 (2.5%)
5,Manager (M4-6),270 (13.4%)
6,Senior Manager (M7),27 (1.3%)
7,Director (M8-9),9 (0.4%)
8,SVP (M11),1 (0.0%)
9,Company Total,"2,014"


### Attrition year-over-year

In [34]:
df_attrition_monthly_yoy = df_monthly.copy(True)
df_attrition_monthly_yoy['yyyy'] = df_attrition_monthly_yoy['ds'].dt.year
df_attrition_monthly_yoy['mm'] = df_attrition_monthly_yoy['ds'].dt.month
df_attrition_monthly_yoy['mmm'] = df_attrition_monthly_yoy['ds'].dt.strftime('%b')

mask_ds = (df_attrition_monthly_yoy.ds_start <= current_month) \
    & (df_attrition_monthly_yoy.ds_start >= '2022-01-01')

df_pivoted = df_attrition_monthly_yoy[mask_ds] \
    [['yyyy', 'mm', 'mmm', 'ds', 'ds_start', 'n_active_employees', 'avg_active_employees', 'attrition_rate']] \
    .pivot_table(index='yyyy', columns='mm', values='attrition_rate')

# create a mapping from month number to abbreviation then rename the columns
month_map = {i: pd.Timestamp(f'2023-{i:02d}-01').strftime('%b') for i in df_pivoted.columns}
df_pivoted = df_pivoted.rename(columns=month_map)

# create formatting for each column then display
# not necessary when displaying into plot
def percent_or_blank(x):
    return '' if pd.isnull(x) else f'{x:.1%}'

formatters = {
    col: percent_or_blank if str(col) else None
    for col in df_pivoted.columns
}

df_pivoted.style.format(formatters)


mm,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
yyyy,,,,,,,,,,,,
2022,3.6%,6.2%,0.0%,6.9%,2.7%,2.9%,3.0%,1.0%,0.0%,2.4%,6.4%,4.0%
2023,1.6%,4.4%,5.5%,5.8%,6.3%,6.3%,4.3%,0.0%,3.4%,3.7%,2.7%,2.5%
2024,2.9%,5.4%,5.5%,6.6%,4.7%,4.3%,0.0%,0.0%,0.8%,4.3%,1.5%,


### Attrition by org

In [35]:
current_month

Timestamp('2024-11-01 00:00:00')

In [36]:
df_employees_plus.columns

Index(['ds_start', 'ds', 'is_last_day_of_year', 'employee_id', 'hire_date',
       'prehire_status', 'is_active', 'termination_date', 'termination_reason',
       'is_termination_voluntary', 'is_terminated', 'org_l00', 'org_l01',
       'org_l02', 'org_l03', 'gender', 'gender_remapped', 'ethnicity',
       'ethnicity_remapped', 'is_manager_track', 'job_track', 'job_level_idx',
       'job_level_category', 'job_level_category_ordered_w_indicators',
       'is_new_hire'],
      dtype='object')

In [37]:
df_monthly_by_org2

,ds,ds_start,org_l00,org_l01,org_l02,n_active_employees,n_new_hire_employees,n_monthly_terminated_employees,avg_active_employees,attrition_rate
0,2021-01-31,2021-01-01,Company Inc.,Administrative,Communications,67,5,3,67.0,0.044776
1,2021-01-31,2021-01-01,Company Inc.,Administrative,Finance,59,1,0,59.0,0.000000
2,2021-01-31,2021-01-01,Company Inc.,Administrative,IT Services,74,6,1,74.0,0.013514
3,2021-01-31,2021-01-01,Company Inc.,Administrative,Legal,57,6,2,57.0,0.035088
4,2021-01-31,2021-01-01,Company Inc.,Administrative,People,66,4,3,66.0,0.045455
...,...,...,...,...,...,...,...,...,...,...
1147,2026-12-31,2026-12-01,Company Inc.,Sales,Business,155,0,0,155.0,0.000000
1148,2026-12-31,2026-12-01,Company Inc.,Sales,Consumer,154,0,0,154.0,0.000000
1149,2026-12-31,2026-12-01,Company Inc.,Sales,Retail,156,0,0,156.0,0.000000
1150,2026-12-31,2026-12-01,Company Inc.,Sales,Sales Operations,177,0,0,177.0,0.000000


In [38]:
# defining orgs that show org_l02 breakdown and order of iteration
org_map = [
    {'org_name': 'Sales', 'incl_org_l02': True},
    {'org_name': 'Production', 'incl_org_l02': True},
    {'org_name': 'Administrative', 'incl_org_l02': False},
]

selected_fields = ['org_l01','org_l02','n_active_employees','n_monthly_terminated_employees','avg_active_employees','attrition_rate']

default_values = {'org_l01': '',
                  'org_l02': '',
                  'n_active_employees': 0,
                  'n_monthly_terminated_employees': 0,
                  'avg_active_employees': 0,
                  'attrition_rate': 0}

attrition_by_org_current_month = pd.DataFrame([default_values])
attrition_by_org_previous_month = pd.DataFrame([default_values])

def prep_display_table(df, ds):
    for org_idx in org_map:
        # calculate values related to org_l02
        if org_idx['incl_org_l02']:
            df = pd.concat([
                df,
                df_monthly_by_org2[(df_monthly_by_org2.ds_start == ds) & (df_monthly_by_org2.org_l01 == org_idx['org_name'])][selected_fields],
            ], ignore_index=True)
        # calculate subtotals that correspond to org_l01
        df = pd.concat([
                df,
                df_monthly_by_org1[(df_monthly_by_org1.ds_start == ds) & (df_monthly_by_org1.org_l01 == org_idx['org_name'])][selected_fields]],
            ignore_index=True
        )
    # calculate grand total
    df = pd.concat([
        df,
        pd.merge(
            left=pd.DataFrame([{'ds': pd.to_datetime(df_monthly[df_monthly.ds_start == ds].ds.iloc[0]), 'ds_start': pd.to_datetime(df_monthly[df_monthly.ds_start == ds].ds_start.iloc[0]), 'org_l01': 'Company Total', 'org_l02': 'Company Total'}]),
            right=df_monthly[df_monthly.ds_start == ds],
            how='left',
            on=['ds', 'ds_start']
        )[selected_fields]
    ], ignore_index=True)
    return df

attrition_by_org_current_month = prep_display_table(attrition_by_org_current_month, current_month)
attrition_by_org_previous_month = prep_display_table(attrition_by_org_previous_month, previous_month)

attrition_by_org_current_month
# attrition_by_org_previous_month


,org_l01,org_l02,n_active_employees,n_monthly_terminated_employees,avg_active_employees,attrition_rate
0,,,0,0,0.0,0.000000
1,Sales,Business,155,2,156.5,0.012780
2,Sales,Consumer,154,3,154.5,0.019417
3,Sales,Retail,156,3,156.5,0.019169
4,Sales,Sales Operations,177,6,178.0,0.033708
5,Sales,Sales Training,158,2,158.5,0.012618
6,Sales,Sales Total,800,16,804.0,0.019900
7,Production,Hardware,133,1,133.5,0.007491
8,Production,Quality Control,107,0,107.5,0.000000
9,Production,Research,109,2,110.0,0.018182


In [39]:
# defining orgs that show org_l02 breakdown and order of iteration
org_l01 = {'Sales': True, 'Production': True, 'Administrative': False}
display_attrition_by_org = pd.DataFrame([{'org_l01': '', 'org_l02': '', 'n_monthly_terminated_employees': 0, 'percent_of_total': 0}])

# filtering data to current month, with corresponding org indices of interest
current_month_headcount_by_org = df_monthly_by_org2[df_monthly_by_org2.ds_start == current_month][['org_l01', 'org_l02', 'n_monthly_terminated_employees']].groupby(by=['org_l01', 'org_l02']).sum().sort_values(by=['org_l01', 'org_l02']).reset_index()
previous_month_headcount_by_org = df_monthly_by_org2[df_monthly_by_org2.ds_start == previous_month][['org_l01', 'org_l02', 'n_monthly_terminated_employees']].groupby(by=['org_l01', 'org_l02']).sum().sort_values(by=['org_l01', 'org_l02']).reset_index()

def prep_display_table(df, dt):
    df['percent_of_total'] = df['n_monthly_terminated_employees'] / df['n_monthly_terminated_employees'].sum()
    
    # creating blank dataframe
    display_attrition_by_org = pd.DataFrame([{'org_l01': '', 'org_l02': '', 'n_monthly_terminated_employees': 0, 'percent_of_total': 0}])
    
    # iterate through orgs determined in the dictionary above, appending subtotals where necessary
    for org in org_l01:
        if org_l01[org]:
            display_attrition_by_org = pd.concat([display_attrition_by_org, df[df.org_l01 == org]], ignore_index=True)
        org_n_terminated_employees = df[df.org_l01 == org].n_monthly_terminated_employees.sum()
        total_terminated_employees = df_employees_plus[(df_employees_plus.ds_start == dt) & (df_employees_plus.is_terminated) & (df_employees_plus['termination_date'].dt.to_period('M') == current_month.to_period('M'))].employee_id.nunique()
        percent_of_total = org_n_terminated_employees / total_terminated_employees
        subtotal_row = {'org_l01': f'{org} Total', 'org_l02': f'{org} Total', 'n_monthly_terminated_employees': org_n_terminated_employees, 'percent_of_total': org_n_terminated_employees / total_terminated_employees}
        display_attrition_by_org = pd.concat([display_attrition_by_org, pd.DataFrame([subtotal_row])], ignore_index=True)

    total_row = {'org_l01': 'Company Total', 'org_l02': 'Company Total', 'n_monthly_terminated_employees': total_terminated_employees, 'percent_of_total': 1}

    # dropping record of blank dataframe
    display_attrition_by_org.drop(index=display_attrition_by_org[(display_attrition_by_org.org_l01 == '') & (display_attrition_by_org.org_l02 == '')].index, axis=0, inplace=True)

    # adding total record
    display_attrition_by_org = pd.concat([display_attrition_by_org, pd.DataFrame([total_row])], ignore_index=True)

    def format_and_combine(row):
        n_active_formatted = f"{row['n_monthly_terminated_employees']:,}"
        if row['org_l02'] == 'Company Total':
            return f"{n_active_formatted}"
        perc_formatted = f"{row['percent_of_total']:.1%}"
        
        return f"{n_active_formatted} ({perc_formatted})"

    # applying formatting to a single column
    display_attrition_by_org['headcount_and_percent_of_total'] = display_attrition_by_org.apply(format_and_combine, axis=1)

    # drop and rename columns
    display_attrition_by_org.rename(columns={'org_l02': 'org'}, inplace=True)
    display_attrition_by_org.drop(columns=['org_l01', 'n_monthly_terminated_employees', 'percent_of_total'], inplace=True)
    
    return display_attrition_by_org

current_attrition_by_org = prep_display_table(current_month_headcount_by_org, current_month)
previous_attrition_by_org = prep_display_table(previous_month_headcount_by_org, previous_month)

current_attrition_by_org
previous_attrition_by_org

/var/folders/0z/lyh3jp3d1y73wqw8jfpcqy_40000gn/T/ipykernel_2687/1942502622.py:21: RuntimeWarning: divide by zero encountered in scalar divide
  percent_of_total = org_n_terminated_employees / total_terminated_employees
/var/folders/0z/lyh3jp3d1y73wqw8jfpcqy_40000gn/T/ipykernel_2687/1942502622.py:22: RuntimeWarning: divide by zero encountered in scalar divide
  subtotal_row = {'org_l01': f'{org} Total', 'org_l02': f'{org} Total', 'n_monthly_terminated_employees': org_n_terminated_employees, 'percent_of_total': org_n_terminated_employees / total_terminated_employees}


,org,headcount_and_percent_of_total
0,Business,7 (8.0%)
1,Consumer,9 (10.3%)
2,Retail,3 (3.4%)
3,Sales Operations,10 (11.5%)
4,Sales Training,6 (6.9%)
5,Sales Total,35 (inf%)
6,Hardware,3 (3.4%)
7,Quality Control,6 (6.9%)
8,Research,7 (8.0%)
9,Service Delivery,7 (8.0%)


In [40]:
current_month.to_period('M')


Period('2024-11', 'M')

In [41]:
df_employees_plus.columns

Index(['ds_start', 'ds', 'is_last_day_of_year', 'employee_id', 'hire_date',
       'prehire_status', 'is_active', 'termination_date', 'termination_reason',
       'is_termination_voluntary', 'is_terminated', 'org_l00', 'org_l01',
       'org_l02', 'org_l03', 'gender', 'gender_remapped', 'ethnicity',
       'ethnicity_remapped', 'is_manager_track', 'job_track', 'job_level_idx',
       'job_level_category', 'job_level_category_ordered_w_indicators',
       'is_new_hire'],
      dtype='object')